In [ ]:
pip install tensorflow-addons

In [ ]:
pip install keras_unet_collection

In [ ]:
import zipfile
from glob import glob
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model, load_model, save_model
import warnings
warnings.filterwarnings('ignore')
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tensorflow.image import resize
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import re
from keras_unet_collection import models, losses
from keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.metrics import Accuracy
from skimage.metrics import structural_similarity as ssim
from tensorflow.keras.layers import Input, BatchNormalization, Dropout, Lambda,add, Conv2D, Conv2DTranspose, MaxPooling2D
from tensorflow.keras.layers import UpSampling2D, concatenate, BatchNormalization, Activation

In [ ]:
data_path = "/kaggle/input/retina-segmentation-data/FIVES/train"

In [ ]:
retina_files = glob(pathname = data_path+'/Original/*')
retina_files[:1]

In [ ]:
mask_files = [path.replace('Original', 'Ground truth') for path in retina_files]
mask_files[:1]

In [ ]:
data_path_test = "/kaggle/input/retina-segmentation-data/FIVES/test"

In [ ]:
retina_files_test = glob(pathname = data_path_test+'/Original/*')
retina_files_test[:1]

In [ ]:
mask_files_test = [path.replace('Original', 'Ground truth') for path in retina_files_test]
mask_files_test[:1]

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i in range(5):
    retina_img = cv2.imread(retina_files[i])
    retina_img_rgb = cv2.cvtColor(retina_img, cv2.COLOR_BGR2RGB)
    axes[i].imshow(retina_img_rgb)
    axes[i].axis('off')
    axes[i].set_title(f'Retina {i+1}')

plt.show()

fig, axes = plt.subplots(1, 5, figsize=(15, 3))

for i in range(5):
    mask_img = cv2.imread(mask_files[i], cv2.IMREAD_GRAYSCALE )
    axes[i].imshow(mask_img, cmap='gray')
    axes[i].axis('off')
    axes[i].set_title(f'Mask {i+1}')

plt.show()

In [ ]:
mask_img[:,:,np.newaxis].shape

In [ ]:
df = pd.DataFrame( data = {
    'retina': retina_files,
    'mask': mask_files
})

In [ ]:
test = pd.DataFrame( data = {
    'retina': retina_files_test,
    'mask': mask_files_test
})

In [ ]:
df

In [ ]:
df.drop(397, inplace=True)

In [ ]:
df_train, df_test = train_test_split(df, test_size = 0.20)

print(df_train.shape)
print(df_test.shape)

In [ ]:
def load_and_preprocess_image(file_path):
    img = cv2.imread(file_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (256,256))
    img_scaled = img_resized.astype(np.float32) / 255.0
    return img_scaled

In [ ]:
def load_and_preprocess_mask(file_path):
    img = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE )
    img = img[:,:,np.newaxis]
    img_resized = cv2.resize(img, (256,256))
    img_scaled = img_resized.astype(np.float32) / 255.0
    img_scaled[img_scaled > 0.5] = 1
    img_scaled[img_scaled <= 0.5] = 0
    return img_scaled

In [ ]:
def generate_data_generator(dataframe, batch_size=8):
    num_samples = len(dataframe)
    while True:
        indices = np.random.randint(0, num_samples, batch_size)
        batch_retina = []
        batch_mask = []
        for idx in indices:
            retina_img = load_and_preprocess_image(dataframe.iloc[idx]['retina'])
            mask_img = load_and_preprocess_mask(dataframe.iloc[idx]['mask'])
            batch_retina.append(retina_img)
            batch_mask.append(mask_img)
        yield np.array(batch_retina), np.array(batch_mask)

In [ ]:
batch_size = 8
train_generator = generate_data_generator(df_train, batch_size=batch_size)
test_generator = generate_data_generator(df_test, batch_size=batch_size)

In [ ]:
def dice_coeff(y_true, y_pred, smooth=100):
    y_true_flatten = K.flatten(y_true)
    y_pred_flatten = K.flatten(y_pred)

    intersection = K.sum(y_true_flatten * y_pred_flatten)
    union = K.sum(y_true_flatten) + K.sum(y_pred_flatten)
    return (2 * intersection + smooth) / (union + smooth)

def dice_coeff_loss(y_true, y_pred, smooth=100):
    return 1 - dice_coeff(y_true, y_pred, smooth)

# Updated Version MyNet

In [ ]:
def encoder_block(inputs, filters):
    conv = Conv2D(filters, kernel_size = (3,3), padding="same")(inputs)
    bn = Activation("relu")(conv)
    conv = Conv2D(filters, kernel_size = (3,3), padding="same")(bn)
    bn = Activation("relu")(conv)
    bn = BatchNormalization(axis=3)(bn)

    return bn;

def decoder_block(inputs, num_filters):
    conv_trans = Conv2DTranspose(num_filters, kernel_size= (3,3), strides = (1,1), padding = "same")(inputs)

    x = Conv2D(num_filters, kernel_size = (3,3), padding= "same")(conv_trans)
    x = Activation("relu")(x)
    x = Conv2D(num_filters, kernel_size = (3,3), padding = "same")(x)   
    x = Activation('relu')(x)
    x = BatchNormalization(axis = 3)(x)

    return x ;

In [ ]:
f = 16
scale = 2

In [ ]:
def mynet_model(input_shape = (None, None, 3)):
    inputs = Input(input_shape)

    block1 = encoder_block(inputs, f)
    
    s1 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(inputs)
    s1 = Activation('relu')(s1)
    
    con1 = concatenate([block1, s1], axis = 3)
    
    block2 = encoder_block(con1, f)
    
    s2 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(inputs)
    s2 = Activation('relu')(s2)
    
    con2 = concatenate([block2, s2], axis = 3)
    
    block3 = encoder_block(con2, f)
    
    s3 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(inputs)
    s3 = Activation('relu')(s3)
    
    con3 = concatenate([block3, s3], axis = 3)
    
    block4 = encoder_block(con3,f)
    
    s4 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(inputs)
    s4 = Activation('relu')(s4)
    
    con4 = concatenate([block4, s4], axis = 3)

    b1 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(con4)
    b1 = Activation('relu')(b1)
    b1 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(b1)  
    b1 = Activation('relu')(b1)
    b1 = BatchNormalization(axis = 3)(b1)
    
    d1 = decoder_block(b1, f)
    
    u1 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(inputs)
    u1 = Activation('relu')(u1)
    
    con5 = concatenate([d1, u1], axis = 3)
    
    d2 = decoder_block(con5, f)
    
    u2 = Conv2D(filters = f, kernel_size = (3,3), padding = 'same')(inputs)
    u2 = Activation('relu')(u2)
    
    con6 = concatenate([d2, u2], axis = 3)
    
    d3 = decoder_block(con6, f)
    
    u3 = Conv2D(filters = f * pow(scale,1), kernel_size = (3,3), padding = 'same')(inputs)
    u3 = Activation('relu')(u3)
    
    con7 = concatenate([d3, u3], axis = 3)
    
    d4 = decoder_block(con7, f)

    outputs = Conv2D(filters = 1, kernel_size = (1,1), activation = "sigmoid")(d4)

    return Model(inputs=[inputs], outputs = [outputs])

In [ ]:
filepath = "/kaggle/working/mynet_1.keras"

checkpoint = ModelCheckpoint(filepath, 
                             monitor='val_dice_coeff',
                             verbose=1, 
                             save_best_only=True, 
                             mode='max')

In [ ]:
model = mynet_model(input_shape = (None,None,3))

In [ ]:
model.summary()

In [ ]:
model.compile(optimizer='adam', loss=dice_coeff_loss, metrics=[dice_coeff])

In [ ]:
epochs=50

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=len(df_train) // batch_size,
    epochs=epochs,
    callbacks=[checkpoint],
    validation_data=test_generator,
    validation_steps=len(df_test) // batch_size
)

In [ ]:
for i in range(20):
    index = np.random.randint(1, len(df_test.index))
    img = cv2.imread(df_test['retina'].iloc[index])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256,256))
    img = img/255
    img = img[np.newaxis, : ,:, :]
    pred_img = model.predict(img)
    
    plt.figure(figsize=(12,12))
    plt.subplot(1, 3, 1)
    plt.imshow(np.squeeze(img))
    plt.title("Original Image")
    plt.subplot(1, 3, 2)
    plt.imshow(np.squeeze(cv2.imread(df_test['mask'].iloc[index])))
    plt.title("Original Mask")
    plt.subplot(1, 3, 3)
    plt.imshow(np.squeeze(pred_img) > 0.5,cmap='gray')
    plt.title("Prediction")
    plt.show()

In [ ]:
for i in range(20):
    index = np.random.randint(1, len(test.index))
    img = cv2.imread(test['retina'].iloc[index])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (256,256))
    img = img/255
    img = img[np.newaxis, : ,:, :]
    pred_img = model.predict(img)
    
    plt.figure(figsize=(12,12))
    plt.subplot(1, 3, 1)
    plt.imshow(np.squeeze(img))
    plt.title("Original Image")
    plt.subplot(1, 3, 2)
    plt.imshow(np.squeeze(cv2.imread(test['mask'].iloc[index])))
    plt.title("Original Mask")
    plt.subplot(1, 3, 3)
    plt.imshow(np.squeeze(pred_img) > 0.5,cmap='gray')
    plt.title("Prediction")
    plt.show()

In [ ]:
test_gen = generate_data_generator(test, batch_size=batch_size)

In [ ]:
def dice_score(model, test_gen, num_samples):
    dice_scores = []
    num_batches = num_samples // batch_size
    
    for _ in range(num_batches):
        retina, mask = next(test_gen)
        predictions = model.predict(retina)
        
        batch_dice_scores = []
        for i in range(batch_size):
            dice_score = dice_coeff(mask[i], predictions[i])
            batch_dice_scores.append(dice_score)
        
        batch_mean_dice = np.mean(batch_dice_scores)
        dice_scores.append(batch_mean_dice)
    
    return np.mean(dice_scores)

In [ ]:
dice_score(model, test_gen, 200)

In [ ]:
model.save('/kaggle/working/best_model.keras')
